# NB 2.3 &mdash; Regressió polinòmica: quan el model aprèn massa

**MP 5134** &mdash; UT2 &mdash; *Versió California Housing*

*Dades: California Housing, districtes censals de Califòrnia (cens de 1990).*

---
### Què farem avui

Al final del NB 2.2 vam topar amb un límit: la latitud i el preu dibuixen dues
muntanyes, i una recta no les pot seguir. Avui aprendrem a **doblegar la recta**.

Però aquest notebook no va realment de corbes. Va del problema més important de
tot l'aprenentatge automàtic, i el veurem amb els nostres ulls: **un model pot
aprendre tan bé les dades d'entrenament que deixi de servir per a dades noves.**

Quan acabis hauries de saber respondre aquestes preguntes:

- Com pot una regressió *lineal* dibuixar una corba?
- Què és el sobreajust i què és l'infraajust?
- Com es detecta el sobreajust, si no el podem dibuixar?
- Què podem fer quan un model sobreajusta?
- I tot això, serveix per predir millor el preu?
- Per què el test segellat no es pot mirar fins al final del curs?

Pel camí trobaràs tres pauses **Ara tu**, com al NB 2.1. Són exercicis curts,
lligats a la secció que acabes de veure, perquè provis amb les teves mans el que
s'acaba d'explicar abans de continuar.

## 1. Les dades, amb el segell posat

Comencem, com hem promès, amb les línies que protegeixen el test segellat. Després
fem la mateixa partició d'entrenament i validació del NB 2.2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

URL_DADES = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
             "master/datasets/housing/housing.csv")
df = pd.read_csv(URL_DADES)

# El test segellat (20 % dels districtes) no es toca fins a la UT10
dades, test_segellat = train_test_split(df, test_size=0.2, random_state=42)
del test_segellat

dades = dades.dropna(subset=["total_bedrooms"])
train, valid = train_test_split(dades, test_size=0.25, random_state=42)
train, valid = train.copy(), valid.copy()

print(f"Districtes d'entrenament: {len(train)}")
print(f"Districtes de validació:  {len(valid)}")

Els mateixos districtes d'entrenament i de validació que al NB 2.2, gràcies als
dos `random_state=42`.

Avui, però, **no predirem el preu**. Recorda com n'era de gruixut el núvol de la
latitud al final del NB 2.2: a la mateixa latitud hi havia districtes de
50.000 \$ i de 500.000 \$. Amb tant de soroll, el que volem veure avui quedaria
tapat. Farem servir una altra relació del mateix fitxer, molt més neta, i
tornarem al preu a la secció 8 i a la latitud a l'exercici 5.

La pregunta d'avui és aquesta: **quina part de les habitacions d'un districte són
dormitoris**, sabent una sola dada, **quantes habitacions té cada llar**. No és un
model gaire útil a la vida real, però té una virtut enorme per aprendre: com que
només té una entrada, tot el que faci el model es pot dibuixar.

Cap de les dues columnes és al fitxer, però les podem fabricar amb les que hi ha,
dividint unes columnes per unes altres.

In [ ]:
for taula in (train, valid):
    taula["habitacions_per_llar"] = taula["total_rooms"] / taula["households"]
    taula["pct_dormitoris"] = 100 * taula["total_bedrooms"] / taula["total_rooms"]

abans = len(train)
train = train[train["habitacions_per_llar"].between(2, 10)].copy()
valid = valid[valid["habitacions_per_llar"].between(2, 10)].copy()

print(f"Districtes d'entrenament que deixem fora: {abans - len(train)}")
print(f"Districtes d'entrenament que ens queden:  {len(train)}")

`habitacions_per_llar` és el total d'habitacions del districte dividit entre el
nombre de llars, i `pct_dormitoris`, el percentatge d'aquestes habitacions que són
dormitoris.

La segona part de la cel·la deixa fora els districtes amb menys de 2 o més de 10
habitacions per llar. Són poc més d'un 1%, i són casos estranys: hi ha districtes
amb desenes d'habitacions per llar, que no s'assemblen a cap casa normal
(probablement zones de segones residències, on moltes cases no tenen ningú que hi
visqui tot l'any). Això vol dir, també, que el model d'avui **només
val per a districtes d'entre 2 i 10 habitacions per llar**. Fora d'aquest marge,
no li podrem demanar res.

## 2. Els dormitoris segons la mida de les llars

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(train["habitacions_per_llar"], train["pct_dormitoris"], alpha=0.15, s=8)
plt.xlabel("Habitacions per llar")
plt.ylabel("Dormitoris (% de les habitacions)")
plt.title("Com més grans les llars, menys part són dormitoris")
plt.show()

Cada punt és un districte. Es veu una forma molt clara: als districtes de llars
petites, de 2 o 3 habitacions, entre un terç i la meitat de les habitacions són
dormitoris. A mesura que les llars creixen, la proporció baixa, primer de pressa i
després cada vegada més a poc a poc, fins a aplanar-se cap al 15%.

Té tot el sentit: un pis petit és un dormitori, una cuina i un menjador. Una casa
gran té tres o quatre dormitoris, i la resta són sala, cuina, menjador,
despatx... Al voltant de la forma hi ha gruix, perquè no totes les cases de cinc
habitacions es reparteixen igual, però molt menys que amb el preu.

## 3. Una recta no pot fer-ho

Abans de res, preparem una columna nova: **la mida de la llar en una escala de 0
a 1**, que és simplement el nombre d'habitacions per llar dividit per 10. És el
mateix número, però en una escala més petita. De seguida veuràs per què ens
convé.

In [ ]:
for taula in (train, valid):
    taula["mida_llar"] = taula["habitacions_per_llar"] / 10

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

recta = LinearRegression().fit(train[["mida_llar"]], train["pct_dormitoris"])

totes_les_mides = pd.DataFrame({"mida_llar": np.linspace(0.2, 1, 200)})
mides_grafica = totes_les_mides["mida_llar"] * 10

plt.figure(figsize=(8, 4))
plt.scatter(train["habitacions_per_llar"], train["pct_dormitoris"], alpha=0.15, s=8)
plt.plot(mides_grafica, recta.predict(totes_les_mides), color="tab:red", linewidth=2)
plt.xlabel("Habitacions per llar")
plt.ylabel("Dormitoris (% de les habitacions)")
plt.show()

print(f"MAE de validació de la recta: {mean_absolute_error(valid['pct_dormitoris'], recta.predict(valid[['mida_llar']])):.2f} punts")

Primer, com es llegeix aquest error. Quan diem que el model s'equivoca **2
punts** volem dir, per exemple, que prediu un 23% de dormitoris en un districte
on n'hi ha un 21%. Tingues present que la mitjana és d'un 21%, i que dir sempre
la mitjana s'equivocaria uns 4 punts.

La recta ho intenta, però no té cap manera de fer una baixada que s'aplana. A les
llars petites i a les grans es queda per sota dels punts, i a les del mig, per
sobre. S'equivoca uns **2 punts** de mitjana.

Aquest problema té nom. Quan un model és **massa simple per a la forma que tenen les
dades**, diem que fa **infraajust** (*underfitting*). L'infraajust es reconeix
perquè el model s'equivoca molt **fins i tot amb les dades d'entrenament**: no és
que no sàpiga generalitzar, és que no ha arribat a aprendre.

## 4. Com doblegar una recta

Aquí ve el truc, i és molt més senzill del que sembla.

Una regressió lineal **només sap sumar columnes multiplicades per un número**. No
la podem canviar. El que sí podem fer és **donar-li columnes noves**, fabricades a
partir de la que ja tenim: la mida de la llar multiplicada per ella mateixa, una
altra vegada per ella mateixa, i així successivament.

Scikit-learn té una eina que fabrica aquestes columnes: `PolynomialFeatures`.
Mirem què fa amb tres districtes d'exemple.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

exemple = pd.DataFrame({"mida_llar": [0.3, 0.5, 0.8]})

fabrica = PolynomialFeatures(degree=3, include_bias=False)
columnes_noves = fabrica.fit_transform(exemple)

pd.DataFrame(columnes_noves, columns=fabrica.get_feature_names_out())

D'**una** columna n'hem fet **tres**:

- `mida_llar`, la original.
- `mida_llar^2`, la mateixa multiplicada per ella mateixa (0,5 × 0,5 = 0,25).
- `mida_llar^3`, multiplicada tres vegades (0,5 × 0,5 × 0,5 = 0,125).

Amb aquestes tres columnes, el model continua fent el mateix de sempre: sumar-les
multiplicades per un número. **Però el resultat ja no és una recta**, perquè
aquestes columnes creixen de maneres diferents i, combinades, poden fer pujades i
baixades. No cal que sàpigues per què funciona; el que importa és veure-ho.

El número `degree` és el **grau**: fins a quantes multiplicacions arribem. Grau 1
és la recta de sempre. Grau 2 permet una corba amb una sola pujada o baixada.
Com més grau, més revolts pot fer la corba.

Ara ja pots entendre per què hem fet servir la mida entre 0 i 1 i no les
habitacions per llar tal qual. Deu habitacions multiplicades quinze vegades per
elles mateixes donen mil bilions (un u amb quinze zeros), i els ordinadors perden
precisió amb números així. La mida de la llar és sempre com a molt 1, i 1 per 1
continua sent 1.

`include_bias=False` evita que la fàbrica afegeixi una columna plena d'uns. No la
necessitem, perquè `LinearRegression` ja té la seva ordenada a l'origen.

Recorda la taula de la UT1: **el grau és un hiperparàmetre**. El model no el tria;
el triem nosaltres. I tota la resta del notebook va de com triar-lo bé.

### 4.1 Corbes de grau 2 i de grau 4

Preparem tres funcions petites: una que fabrica les columnes i entrena el model,
una que fa prediccions i una que en calcula l'error. Les farem servir moltes
vegades.

In [ ]:
def entrenar_polinomi(grau, dades_entrenament):
    """Fabrica les columnes polinòmiques i entrena una regressió lineal amb elles."""
    fabrica = PolynomialFeatures(degree=grau, include_bias=False)
    X = fabrica.fit_transform(dades_entrenament[["mida_llar"]])
    model = LinearRegression().fit(X, dades_entrenament["pct_dormitoris"])
    return fabrica, model


def predir_polinomi(fabrica, model, taula):
    return model.predict(fabrica.transform(taula[["mida_llar"]]))


def mae_polinomi(fabrica, model, taula):
    return mean_absolute_error(taula["pct_dormitoris"], predir_polinomi(fabrica, model, taula))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for eix, grau in zip(eixos, [2, 4]):
    fabrica, model = entrenar_polinomi(grau, train)
    eix.scatter(train["habitacions_per_llar"], train["pct_dormitoris"], alpha=0.15, s=8)
    eix.plot(mides_grafica, predir_polinomi(fabrica, model, totes_les_mides), color="tab:red", linewidth=2)
    eix.set_title(f"Grau {grau}: MAE de validació {mae_polinomi(fabrica, model, valid):.2f} punts")
    eix.set_xlabel("Habitacions per llar")

eixos[0].set_ylabel("Dormitoris (% de les habitacions)")
plt.show()

El grau 2 ja fa la corba: baixa de pressa a l'esquerra i s'aplana a la dreta, i
l'error passa de 2 punts a menys d'1,5. El grau 4 hi afegeix poc: segueix una
mica millor les llars més petites, a l'esquerra de tot, on la baixada és més
forta, però l'error gairebé no es mou. Aquesta forma és senzilla, una sola baixada
que es va aplanant, i per a això n'hi ha prou amb un sol revolt.

Fins aquí, tot són bones notícies: més grau, millor model, o com a mínim no
pitjor. Anem a veure on s'acaba la festa.

---
### Ara tu · Exercici 1: entrenament contra validació

Al NB 2.1 vas comparar l'error d'un model amb les dades d'entrenament i amb les de
prova. Fes el mateix aquí: amb `mae_polinomi`, calcula el MAE d'entrenament i el
de validació dels graus 1, 2 i 4 entrenats amb `train`, i posa'ls en una taula.

Els dos errors de cada grau s'assemblen o són molt diferents? I la recta,
s'equivoca molt només amb els districtes nous o també amb els que ha vist?
Guarda't la resposta, perquè d'aquí a una estona la necessitaràs.

Pista: `entrenar_polinomi(1, train)` també funciona. Amb grau 1, la fàbrica deixa
la columna tal com és i surt la recta de sempre.

In [ ]:
# Escriu aquí el teu codi

## 5. Quan tenim poques dades

Fins ara hem entrenat amb més de dotze mil districtes. Imagina ara que només en
tenim **quinze**. És una situació molt més habitual del que sembla: moltes
vegades les dades són cares o escasses, i un cens com aquest costa molts diners i
es fa una vegada cada deu anys.

In [ ]:
poques = train.sample(15, random_state=27)

plt.figure(figsize=(8, 4))
plt.scatter(poques["habitacions_per_llar"], poques["pct_dormitoris"], s=50, color="black")
plt.xlabel("Habitacions per llar")
plt.ylabel("Dormitoris (% de les habitacions)")
plt.title("Només quinze districtes")
plt.xlim(2, 10)
plt.show()

Quinze punts. Es continua intuint la baixada, però hi ha forats grans: tots els
districtes tenen entre 3,8 i 7,2 habitacions per llar. **No en tenim cap de llars
petites ni cap de llars grans.**

Entrenem-hi tres models: un de massa simple, un de raonable i un de molt complex.
Per a cadascun mirarem **dos errors**: el que fa amb aquests mateixos quinze
districtes i el que fa amb els més de quatre mil de validació.

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for eix, grau in zip(eixos, [1, 2, 15]):
    fabrica, model = entrenar_polinomi(grau, poques)
    error_train = mae_polinomi(fabrica, model, poques)
    error_valid = mae_polinomi(fabrica, model, valid)

    eix.scatter(valid["habitacions_per_llar"], valid["pct_dormitoris"], alpha=0.1, s=6, color="gray", label="validació")
    eix.scatter(poques["habitacions_per_llar"], poques["pct_dormitoris"], s=40, color="black", label="els 15 districtes")
    eix.plot(mides_grafica, predir_polinomi(fabrica, model, totes_les_mides), color="tab:red", linewidth=2)
    eix.set_ylim(0, 60)
    eix.set_title(f"Grau {grau}\nentrenament {error_train:.2f} punts · validació {error_valid:.1f} punts")
    eix.set_xlabel("Habitacions per llar")

eixos[0].set_ylabel("Dormitoris (% de les habitacions)")
eixos[0].legend(loc="upper right")
plt.show()

Aquestes tres gràfiques són el centre de tot el notebook. Mira-les amb calma.

**Grau 1.** La recta de sempre. Amb els quinze districtes no s'equivoca gaire,
perquè tots són a la zona del mig, on la corba és gairebé recta. Però amb la
validació s'equivoca gairebé 2 punts, com la recta de la secció 3: no té la forma
de les dades. Infraajust.

**Grau 2.** La corba passa a prop dels punts sense tocar-los tots i dibuixa una
baixada creïble. S'equivoca menys d'un punt amb els quinze districtes i un punt i
mig amb la validació, gairebé igual que amb les dotze mil dades. A l'extrem dret,
on no té cap punt, torna a pujar una mica més del compte, però res de greu. És un
model raonable.

**Grau 15.** Aquí passa una cosa sorprenent. **La corba passa pràcticament per
sobre dels quinze punts**: l'error d'entrenament és d'unes dècimes, gairebé
perfecte. I en canvi, entre punt i punt i sobretot als extrems fa **disbarats**:
puja i baixa de manera salvatge i surt del gràfic. L'error de validació es compta
per milions de punts, en una cosa que només pot anar de 0 a 100.

Hem hagut de retallar l'eix vertical perquè la gràfica es pogués llegir. Mira fins
on arriba de veritat:

In [ ]:
fabrica_15, model_15 = entrenar_polinomi(15, poques)
prediccions_15 = predir_polinomi(fabrica_15, model_15, totes_les_mides)

print(f"Percentatge de dormitoris més baix que prediu el grau 15: {prediccions_15.min():,.0f} %")
print(f"Percentatge de dormitoris més alt que prediu el grau 15:  {prediccions_15.max():,.0f} %")

Un percentatge de dormitoris de **menys mil milions** per a les llars de deu
habitacions, i de més del 100% per a unes altres, en una cosa que per força ha
d'estar entre el 0 i el 100%. Els disbarats són justament als llocs on no hi havia
cap dels quinze punts per retenir la corba, sobretot les llars petites i les
grans. I, alhora, és el model que millor s'ajusta a les dades d'entrenament.

Això és el **sobreajust** (*overfitting*): el model té tanta llibertat que, en lloc
d'aprendre la forma general de les dades, **les memoritza**, soroll inclòs. Els
quinze punts tenen petites irregularitats de districtes concrets que no es tornaran
a repetir, i el grau 15 ha doblegat la corba per passar exactament per totes
elles. Entre punt i punt, i més enllà de l'últim, no hi ha res que el retingui.

És com l'alumne que es prepara un examen memoritzant les respostes del de l'any
passat: ho clava si li posen les mateixes preguntes, i no sap fer res si canvien
una coma.

---
### Ara tu · Exercici 2: les llars petites

Amb els quinze districtes d'aquesta secció (`poques`), fes que el model de grau 2
i el de grau 15 prediguin el percentatge de dormitoris d'un districte de **llars
de tres habitacions** (mida 0,3). Compara-ho amb el percentatge real mitjà dels
districtes d'entrenament que tenen entre 2,5 i 3,5 habitacions per llar. Per què el
grau 15 falla tant precisament aquí? Torna a mirar la gràfica dels quinze punts.

Pista: per predir un sol districte, les funcions necessiten una taula amb la
columna `mida_llar`, com ara `pd.DataFrame({"mida_llar": [0.3]})`. Per al valor
real, `between(2.5, 3.5)` et va bé per filtrar.

In [ ]:
# Escriu aquí el teu codi

## 6. Com es detecta el sobreajust sense dibuixar-lo

Amb una sola variable hem pogut dibuixar la corba i veure-hi els disbarats. Però
els models de veritat tenen deu, cent o mil variables, i no es poden dibuixar. Com
sabem, llavors, si un model sobreajusta?

La resposta és a les dues columnes que hem estat mirant: **l'error d'entrenament i
l'error de validació**. Fem-ho per a tots els graus de l'1 al 15.

In [ ]:
escombrada = []
for grau in range(1, 16):
    fabrica, model = entrenar_polinomi(grau, poques)
    escombrada.append({
        "grau": grau,
        "MAE entrenament": mae_polinomi(fabrica, model, poques),
        "MAE validació": mae_polinomi(fabrica, model, valid),
    })

escombrada = pd.DataFrame(escombrada)
escombrada.round(2)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(escombrada["grau"], escombrada["MAE entrenament"], marker="o", label="entrenament (els 15 districtes)")
plt.plot(escombrada["grau"], escombrada["MAE validació"], marker="o", label="validació")
plt.ylim(0, 5)
plt.xticks(range(1, 16))
plt.xlabel("Grau del polinomi")
plt.ylabel("MAE (punts)")
plt.legend()
plt.title("L'error d'entrenament sempre baixa; el de validació, no")
plt.show()

Aquesta gràfica, amb aquesta forma, la tornaràs a veure moltes vegades durant el
curs. Fixa-t'hi bé.

**La línia d'entrenament només baixa.** Cada grau més dóna més llibertat a la corba
per acostar-se als quinze punts, i per tant l'error d'entrenament sempre millora.
Si només miréssim aquesta línia, triaríem el grau 15 i ens equivocaríem de ple.

**La línia de validació fa una vall.** Al principi baixa, perquè el model passa de
massa simple a raonable. Arriba a un mínim, aquí al grau 2. I a partir d'aquí
torna a pujar, primer poc i després de manera desbocada: el model ha començat a
memoritzar. (Hem retallat l'eix a 5 punts; els últims graus se'n van a milions.)

D'aquí surt la regla per detectar-ho:

| Què veus | Diagnòstic |
|---|---|
| Error alt a l'entrenament **i** a la validació | **Infraajust**: el model és massa simple |
| Error baix a l'entrenament **i** a la validació, semblants | Model raonable |
| Error baix a l'entrenament però **molt més alt** a la validació | **Sobreajust**: el model memoritza |

**El senyal d'alarma és la distància entre les dues línies.** Recorda el NB 2.2: la
regressió lineal amb vuit variables tenia errors d'entrenament i de validació
gairebé iguals. Ara ja saps per què això era una bona notícia.

I aquest és el motiu de fons pel qual existeix el conjunt de validació: **per
triar els hiperparàmetres**, com el grau, sense fer trampa amb el test. Però
triar també té un preu, i el veurem a la secció 9.

---
### Ara tu · Exercici 3: uns altres quinze districtes

Repeteix l'escombrada d'aquesta secció triant uns altres quinze districtes: canvia
`random_state=27` per `0`, `3` i `7`. El millor grau és sempre el 2? Què et diu
això sobre la confiança que podem tenir en una tria feta amb tan poques dades?

Pista: desa els districtes nous en una variable amb un altre nom (per exemple,
`altres`), perquè `poques` la tornarem a necessitar a la secció 7.

In [ ]:
# Escriu aquí el teu codi

## 7. I si tinguéssim més dades?

Tornem al grau 15, el model desbocat. Què passa si, en lloc de quinze districtes,
li donem els més de dotze mil d'entrenament?

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for eix, (nom, taula) in zip(eixos, [("15 districtes", poques), ("tots els districtes d'entrenament", train)]):
    fabrica, model = entrenar_polinomi(15, taula)
    eix.scatter(taula["habitacions_per_llar"], taula["pct_dormitoris"], alpha=0.3 if len(taula) > 100 else 1,
                s=8 if len(taula) > 100 else 40, color="black")
    eix.plot(mides_grafica, predir_polinomi(fabrica, model, totes_les_mides), color="tab:red", linewidth=2)
    eix.set_ylim(0, 60)
    eix.set_title(f"Grau 15 amb {nom}\nentrenament {mae_polinomi(fabrica, model, taula):.2f} punts · "
                  f"validació {mae_polinomi(fabrica, model, valid):.2f} punts")
    eix.set_xlabel("Habitacions per llar")

eixos[0].set_ylabel("Dormitoris (% de les habitacions)")
plt.show()

**El mateix grau 15, amb moltes dades, es comporta bé.** La corba segueix la
baixada i l'error de validació baixa per sota d'1,5 punts, com el del grau 4 de la
secció 4.1. Els errors d'entrenament i de validació tornen a ser semblants. Només
fa una petita ona a l'extrem dret, que és on hi ha menys districtes: el sobreajust
sempre comença pels llocs amb poques dades.

Què ha passat? Amb milers de punts repartits per totes les mides de llar, **no hi
ha forats on la corba es pugui desbocar**, i cap punt sol no la pot estirar cap a
ell perquè n'hi ha molts altres al voltant que la retenen.

La conclusió és important: **el sobreajust no depèn només del model, depèn de la
relació entre la complexitat del model i la quantitat de dades.** Un model massa
complex per a quinze dades pot ser perfectament raonable per a dotze mil.

Quan un model sobreajusta, doncs, hi ha dues sortides:

- **Fer el model més simple** (aquí, baixar el grau).
- **Aconseguir més dades**, quan es pot.

Veuràs aquesta mateixa història amb altres models. A la **UT5**, el grau es dirà
*profunditat de l'arbre*. A la **UT6**, el paràmetre `C` de les SVM. A la **UT7**
aprendràs a triar aquests hiperparàmetres d'una manera molt més fiable que amb
una sola partició de validació.

## 8. I el preu?

Fins ara hem fet servir un problema de joguina, els dormitoris segons la mida de
les llars, perquè es podia dibuixar. Però el problema de veritat és el del NB 2.2:
**predir el preu**. Allà vam passar de 63.000 \$ d'error amb els ingressos sols a
51.000 \$ amb les vuit variables, i vam veure que la latitud dibuixava dues
muntanyes que una recta no podia seguir.

Ara tenim una eina nova. Tornem-hi. Primer recuperem les dades del NB 2.2: tornem
a partir `dades` amb el mateix `random_state=42`, que ens dóna els mateixos
districtes d'entrenament i de validació, sense el filtre de les habitacions per
llar. El segell continua posat.

In [ ]:
FEATURES = ["median_income", "latitude", "longitude", "housing_median_age",
            "total_rooms", "total_bedrooms", "population", "households"]
OBJECTIU = "median_house_value"

train_preu, valid_preu = train_test_split(dades, test_size=0.25, random_state=42)
train_preu, valid_preu = train_preu.copy(), valid_preu.copy()
for taula in (train_preu, valid_preu):
    taula["posicio"] = (taula["latitude"] - 32) / 10


def provar(nom, X_train, X_valid):
    """Entrena una regressió lineal amb les columnes donades i en mesura els dos errors."""
    model = LinearRegression().fit(X_train, train_preu[OBJECTIU])
    return {
        "model": nom,
        "columnes": X_train.shape[1],
        "MAE entrenament": mean_absolute_error(train_preu[OBJECTIU], model.predict(X_train)),
        "MAE validació": mean_absolute_error(valid_preu[OBJECTIU], model.predict(X_valid)),
    }


print(f"Districtes d'entrenament: {len(train_preu)}")
print(f"Districtes de validació:  {len(valid_preu)}")

Són els mateixos districtes que al NB 2.2. La columna `posicio` és la latitud
posada en una escala petita, com vam fer amb la mida de la llar: els graus per
sobre del paral·lel 32, dividits per 10.

La funció `provar` ens estalviarà repetir codi: rep les columnes d'entrenament i
les de validació, entrena una regressió lineal i torna els dos errors, a més del
nombre de columnes que ha fet servir el model.

Comencem per la idea que va quedar pendent: afegir a les vuit variables del NB 2.2
**la corba de la latitud**, amb prou grau perquè pugui fer les dues muntanyes.

In [ ]:
corba_latitud = PolynomialFeatures(degree=8, include_bias=False)
latitud_train = corba_latitud.fit_transform(train_preu[["posicio"]])
latitud_valid = corba_latitud.transform(valid_preu[["posicio"]])

comparacio = pd.DataFrame([
    provar("ingressos sols (NB 2.2)", train_preu[["median_income"]], valid_preu[["median_income"]]),
    provar("vuit variables (NB 2.2)", train_preu[FEATURES], valid_preu[FEATURES]),
    provar("vuit variables + corba de la latitud de grau 8",
           np.hstack([train_preu[FEATURES], latitud_train]),
           np.hstack([valid_preu[FEATURES], latitud_valid])),
])
comparacio.round(0)

`np.hstack` enganxa les columnes una al costat de l'altra: les vuit variables i
les vuit de la corba, setze en total.

La corba de la latitud ajuda, però poc: l'error baixa uns 1.100 \$, de 51.200 a
50.100 \$. És el que esperàvem després del final del NB 2.2. Les dues muntanyes hi
són, però al voltant hi ha tant de soroll que encertar la forma de la corba es nota
poc. A més, el model ja sabia bastant d'on era cada districte gràcies a la
longitud i la latitud juntes.

### 8.1 Combinar les variables entre elles

`PolynomialFeatures` encara pot fer una cosa més. Si li donem **diverses columnes**
en lloc d'una, no només fabrica els quadrats de cadascuna: també les **multiplica
entre elles**, de dues en dues. Amb grau 2 i les vuit variables surten 44 columnes:
les vuit originals, els vuit quadrats i els 28 productes de parelles, com
`median_income × latitude` o `total_rooms × households`. Aquests productes
permeten al model aprendre coses com que *uns ingressos alts no pesen igual a Los
Angeles que al nord rural*.

Fem l'escombrada de la secció 6, però ara amb el problema de veritat.

In [ ]:
escombrada_preu = []
for grau in (1, 2, 3, 4):
    fabrica = PolynomialFeatures(degree=grau, include_bias=False)
    X_train = fabrica.fit_transform(train_preu[FEATURES])
    X_valid = fabrica.transform(valid_preu[FEATURES])
    escombrada_preu.append(provar(f"vuit variables, grau {grau}", X_train, X_valid))

pd.DataFrame(escombrada_preu).round(0)

Quatre files, i hi ha tota la història del notebook:

- **Grau 1** són les vuit variables del NB 2.2, amb els dos errors gairebé iguals.
- **Grau 2**, amb 44 columnes, baixa l'error de validació fins a uns **45.700 \$**,
  5.500 \$ menys que el grau 1. Això sí que és una millora de veritat: els productes
  entre variables aporten informació que una simple suma no podia veure.
- **Grau 3**, amb 164 columnes, encara baixa una mica la validació, fins a uns
  45.000 \$. Però mira la distància entre les dues columnes: ha passat d'uns
  1.100 \$ a uns 3.100 \$.
- **Grau 4**, amb 494 columnes, és el sobreajust de la secció 5, però ara amb el
  problema de veritat. L'error de validació **puja** fins a 49.900 \$, gairebé com
  el del grau 1, i la distància amb l'entrenament ja és de més de 8.000 \$. És el
  senyal d'alarma de la secció 6.

(Potser et sorprèn que l'error d'entrenament del grau 4 gairebé no baixi. És el
problema de la secció 4: `total_rooms` o `population`, elevats a la quarta,
donen números enormes, i l'ordinador hi perd precisió. A la **UT3** aprendràs a
posar totes les variables a la mateixa escala, que és com es resol.)

Fixa't que aquí tenim més de dotze mil dades, i tot i així el grau 4 sobreajusta.
És la secció 7 vista des de l'altre costat: **dotze mil dades són moltes per a una
corba de quinze coeficients, però poques per a un model de 494.**

Quin grau triaries? El grau 3 té l'error de validació més baix, però el grau 2 hi
queda a tocar amb molta menys distància entre els dos errors i amb quatre vegades
menys columnes. Si tries el 2, estàs aplicant el que has après avui.

Resumint: la regressió múltiple ja millorava molt el preu al NB 2.2, i combinant
les variables encara millora més. Amb el grau 2 o el 3, el model s'equivoca gairebé
un 30% menys que amb els ingressos sols. A la **UT3** aprendràs a fabricar a mà
columnes com aquestes, triant les que tenen sentit (com les habitacions per llar)
en lloc de generar-les totes.

## 9. Ara sí: per què el test està segellat

Al NB 2.2 vam apartar un 20% dels districtes i et vam prometre que entendries per
què. Acabes de fer exactament el que ho explica.

A la secció 6 has entrenat quinze models, has mirat l'error de validació de
cadascun i t'has quedat amb el que en tenia menys. És el que s'ha de fer. Però
pensa què vol dir: **el grau 2 ha guanyat perquè era el que millor anava amb
aquestes dades concretes**. Amb uns altres quinze districtes d'entrenament, el
guanyador podria haver estat l'1 o el 3 (ho has comprovat a l'exercici 3). L'error
del grau 2 és l'error del model que ha quedat primer en una competició, i al
primer lloc sempre hi ha una mica de sort. És un número lleugerament optimista. I
el mateix val per als 45.000 \$ de la secció 8: també són el guanyador d'una
competició.

Una tria sola no fa gaire mal. El problema és que durant el curs en farem
centenars: aquesta variable sí, aquesta no, aquest grau, aquest model és millor
que aquell... Cada decisió que prenem mirant la validació **adapta una mica el
nostre treball a aquells districtes concrets**. Al cap de tres mesos, el número de
validació ja no serà una estimació honesta de com funcionarà el model amb
districtes nous. És una trampa que no es veu, perquè cap decisió sola és una
trampa: és la suma de totes.

Per això necessitem un tercer tros que **no hagi participat en cap decisió**, i
aquest és el test segellat. Completem la taula del NB 2.2 amb una columna nova:

| Tros | Per a què | Quantes vegades el mirem |
|---|---|---|
| **Entrenament** (60 %) | el model hi aprèn amb `fit()` | cada vegada que entrenem |
| **Validació** (20 %) | nosaltres hi comparem models i decidim | tantes com calgui |
| **Test segellat** (20 %) | comprovar, al final, si tot plegat funciona | **una sola vegada**, a la UT10 |

Si algun dia miréssim el test *només per curiositat* i després canviéssim alguna
cosa, encara que fos sense voler, el test passaria a ser una validació més i ja no
tindríem cap manera honesta de saber com funcionen els nostres models. Per això
tots els notebooks comencen amb el mateix `train_test_split` i el mateix
`random_state=42`: no és desconfiança, és l'única manera que el número final
valgui alguna cosa.

## 10. Resum

- Una regressió lineal pot dibuixar corbes si li fabriquem columnes noves amb
  `PolynomialFeatures`. El **grau** és un hiperparàmetre que decideix quants
  revolts pot fer la corba.
- **Infraajust**: el model és massa simple i s'equivoca molt fins i tot amb les
  dades d'entrenament.
- **Sobreajust**: el model memoritza les dades d'entrenament i s'equivoca molt
  amb les noves.
- Es detecten comparant l'**error d'entrenament** amb l'**error de validació**.
  Una distància gran entre tots dos és el senyal d'alarma.
- L'error d'entrenament **no serveix per triar el model**: sempre prefereix el més
  complex.
- El sobreajust depèn de la complexitat del model **i** de la quantitat de dades.
- Amb el problema de veritat, les columnes polinòmiques milloren la predicció
  sobretot quan combinen variables. I amb massa columnes, també sobreajusten.
- La validació serveix per decidir, i com que hi decidim tantes vegades, acaba
  sent una mica optimista. Per això el **test segellat** no participa en cap
  decisió i només s'obre una vegada, a la UT10.

## 11. Exercicis finals

Els tres exercicis *Ara tu* els has fet pel camí, cadascun lligat a una secció.
Aquests quatre últims et demanen fer servir el notebook sencer.

**4. Quaranta districtes.** Repeteix la secció 6 amb 40 districtes en lloc de 15
(`train.sample(40, random_state=1)`). On és ara el mínim de la validació? El
grau 15 és tan desastrós com abans?

**5. Tornem a la latitud.** Repeteix la secció 4.1 amb la relació del final del
NB 2.2: el preu (`median_house_value`) a partir de la latitud, amb totes les dades
d'entrenament. Per mantenir l'escala petita, fes servir `(latitude - 32) / 10`
en lloc de `mida_llar`. Prova els graus 1, 2, 4, 8 i 12. A partir de quin grau la
corba troba les dues muntanyes, Los Angeles i San Francisco? Quant baixa el MAE
respecte a la recta, en percentatge? I amb els dormitoris, quant baixava? Per què
creus que la corba ajuda tan poc amb el preu?
Pista: hauràs d'adaptar les funcions, que tenen `"mida_llar"` i
`"pct_dormitoris"` escrits a dins. Recorda que amb el preu l'error surt en dòlars.

**6. Per escrit.** Explica amb les teves paraules, en un paràgraf, la diferència
entre un model que *aprèn* i un model que *memoritza*. Fes servir com a exemple
alguna situació de fora de la informàtica que no sigui la de l'examen.

**7. Per escrit.** Un company proposa: *"Només per curiositat, calculem el MAE del
model sobre el test segellat. No canviarem res, només mirarem."* Explica en unes
quatre línies per què no s'ha de fer, fent servir el que has après a la secció 9.

## 12. Per al debat de classe

- Un company us ensenya un model amb un R2 de 0,99 sobre les dades d'entrenament.
  Quina és la primera pregunta que li faríeu?
- Penseu en situacions reals on **només hi ha poques dades** i on, per tant, el
  risc de sobreajust és alt: un producte que fa un mes que es ven, una malaltia
  rara, una botiga que acaba d'obrir... Què faríeu en cada cas?
- Sempre és possible *aconseguir més dades*? Quins costos té? Penseu en el cens
  d'on surten aquestes dades.